# DPO round -- qwen3.5-4b extraction model (Colab)

Starts from the SFT checkpoint `opus-distill-2026-10-ckpt100` and runs one round of
Direct Preference Optimisation against the student's OWN unsupported facts, then pushes
the merged model as a new revision of `carev01/qwen35-4b-graphrag`.

**Why (2026-10-21).** The teacher judge showed Opus's training targets are clean (0.8%
unsupported on held-out chunks). The unsupported facts the distilled model writes come
from the student itself, some from pretraining knowledge of the vendor docs. SFT only
ever shows good outputs, so it cannot train that out. DPO shows the model its own reply
with the invented fact (`rejected`) next to the same reply without it (`chosen`).

**The dataset** comes from `scripts/distill/dpo_pairs.py`. ckpt100 served on vLLM
extracted 700 fresh articles, none of them in the distillation or trial samples
(`haiku_extract_ab.py --capture`). Each edge reply was judged twice, and a fact counts as
unsupported only when both passes agree. `chosen` differs from `rejected` only by those
edges, so preference moves away from the invented facts and not from extraction in
general.

**Guards against the known DPO failure modes:**
- *Losing coverage*: an NLL term on `chosen` (`NLL_ALPHA`: TRL's `rpo_alpha`, or its `"sft"` loss) keeps the
  likelihood of the good edges up. The end-to-end trial gates on distinct facts too.
- *Broken JSON*: guided decoding enforces the schema at serving time. This notebook also
  smoke-tests three held-out prompts before pushing.
- *Training a different completion than vLLM serves*: section 8 decodes the trainer's own
  tokenised row and checks it equals serving prompt + answer + `<|im_end|>`.
- *Overfitting* (v2 overfit after step 100): the best checkpoint by eval loss is kept
  (`load_best_model_at_end`), and checkpoints are saved at every eval so it is never
  rotated out.

**Runtime:** L4 or A100. Edge prompts run to several thousand tokens, and DPO runs each
pair's chosen, rejected and reference passes. It **never overwrites** a revision that
already holds weights.

**Use:** set the HF token secret (section 2), check section 3, then **Runtime -> Run all**.
A dead runtime resumes from the last checkpoint on Drive.

## 1. Install packages

Same install as `distill_train.ipynb` section 1.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def pip_install(*args: str) -> None:
    """Run pip, print ALL of its output, and raise on failure (never hidden)."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout, flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"pip install failed (exit {proc.returncode}): {' '.join(args)}")


if importlib.util.find_spec("unsloth") is None:
    pip_install("--upgrade", "pip")
    pip_install("unsloth")
    pip_install("trl", "datasets", "huggingface_hub")

for pkg in ("unsloth", "trl", "transformers", "peft", "datasets", "huggingface_hub"):
    print(f"{pkg}=={importlib.metadata.version(pkg)}")
print("packages ready")

## 1a. Fast causal_conv1d kernel (best effort)

The Gated DeltaNet layers run a short causal convolution. Without the `causal_conv1d`
package transformers uses its reference PyTorch version ("correct but much slower"; dpo1
ran that way, ~2.5 min per step on an L4). PyPI ships only a source build and the
package's GitHub wheels stop at torch 2.10, but the HF kernel hub
(`kernels-community/causal-conv1d`) has a prebuilt build for this exact torch/CUDA.

This cell loads that build and checks it against the package's own reference
implementation (forward, backward and the one-token decode step). It then registers it
as `causal_conv1d` **before** transformers imports the Qwen3.5 modelling code, which looks
the package up once, at import time. Any failure leaves the PyTorch path in place, so
the run continues, only slower. Section 6 then confirms which implementation the
loaded model actually calls.

In [ ]:
import importlib.util
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

CC1D_REPO, CC1D_REVISION = "kernels-community/causal-conv1d", "v1"
_QWEN_MOD = "transformers.models.qwen3_5.modeling_qwen3_5"


def load_causal_conv1d():
    tv = "".join(torch.__version__.split("+")[0].split(".")[:2])
    variant = f"torch{tv}-cxx11-cu{torch.version.cuda.replace('.', '')}-x86_64-linux"
    root = Path(snapshot_download(CC1D_REPO, revision=CC1D_REVISION,
                                  allow_patterns=[f"build/{variant}/*"]))
    pkg = root / "build" / variant
    if not (pkg / "__init__.py").exists():
        raise FileNotFoundError(f"{CC1D_REPO}@{CC1D_REVISION} has no {variant} build")
    spec = importlib.util.spec_from_file_location(
        "causal_conv1d", pkg / "__init__.py", submodule_search_locations=[str(pkg)])
    mod = importlib.util.module_from_spec(spec)
    sys.modules["causal_conv1d"] = mod
    spec.loader.exec_module(mod)
    return mod, variant


def check_causal_conv1d(mod) -> None:
    # Compare against the reference implementations shipped in the same package.
    ref = mod.causal_conv1d_interface
    g = torch.Generator(device="cuda").manual_seed(0)
    x = torch.randn(2, 96, 333, device="cuda", dtype=torch.bfloat16, generator=g)
    w = torch.randn(96, 4, device="cuda", dtype=torch.bfloat16, generator=g)
    b = torch.randn(96, device="cuda", dtype=torch.bfloat16, generator=g)
    xk, xr = x.clone().requires_grad_(), x.float().clone().requires_grad_()
    out = mod.causal_conv1d_fn(xk, w, b, activation="silu")
    want = ref.causal_conv1d_ref(xr, w.float(), b.float(), activation="silu")
    assert torch.allclose(out.float(), want, atol=0.1, rtol=0.05), "forward mismatch"
    out.float().sum().backward()
    want.sum().backward()
    assert torch.allclose(xk.grad.float(), xr.grad, atol=0.1, rtol=0.05), "backward mismatch"
    state = torch.randn(2, 96, 4, device="cuda", dtype=torch.bfloat16, generator=g)
    step = torch.randn(2, 96, 1, device="cuda", dtype=torch.bfloat16, generator=g)
    s_k, s_r = state.clone(), state.float().clone()
    out = mod.causal_conv1d_update(step, s_k, w, b, activation="silu")
    want = ref.causal_conv1d_update_ref(step.float(), s_r, w.float(), b.float(), activation="silu")
    assert torch.allclose(out.float(), want, atol=0.1, rtol=0.05), "decode-step mismatch"
    assert torch.allclose(s_k.float(), s_r, atol=0.05), "decode-step state mismatch"


CC1D_LOADED = False
if _QWEN_MOD in sys.modules:
    print("transformers' Qwen3.5 code is already imported, so a kernel registered now would "
          "be ignored -- Runtime -> Restart session, then Run all")
else:
    try:
        _cc1d, _variant = load_causal_conv1d()
        check_causal_conv1d(_cc1d)
        CC1D_LOADED = True
        print(f"causal_conv1d: {CC1D_REPO} {_variant} loaded and matches its reference")
    except Exception as e:
        for _k in [k for k in sys.modules if k.split(".")[0] == "causal_conv1d"]:
            del sys.modules[_k]
        print(f"causal_conv1d: hub kernel NOT used ({type(e).__name__}: {e}) -- "
              "training runs on the PyTorch path (correct, slower)")

## 2. HF token -- Colab secret or environment, never hardcoded

Set a Colab secret named `HF_TOKEN` (the key icon in the left sidebar) *or* export
`HF_TOKEN` in the environment before starting this runtime. This cell never prints the
token -- only whether one was found.

In [ ]:
import os


def get_hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok
        except Exception:
            pass  # secret not set -- fall through to the environment
    except ImportError:
        pass  # not running on Colab
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        raise RuntimeError(
            "no HF token found -- add a Colab secret named HF_TOKEN, or set the "
            "HF_TOKEN environment variable, before running this cell")
    return tok


HF_TOKEN = get_hf_token()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF token loaded and logged in (value never printed)")

## 3. Configuration

The DPO hyper-parameters are starting points, not tuned values: the held-out eval and the
end-to-end trial decide. **Learning rate 5e-5, not 5e-6 (2026-10-22).** The dpo1 run at
5e-6 separated the val pairs (reward accuracy 0.97) but moved them too little to change
what the model writes. At the point where `chosen` and `rejected` split, the mean log-odds
of skipping the unsupported edge went from -3.29 to -2.95, and greedy decoding skipped it
in 4 of 39 pairs both before and after. A LoRA that starts at zero needs roughly ten times
a full fine-tune's DPO rate. Section 10 re-runs that check as an export gate.

In [ ]:
from pathlib import Path

# --- dataset: dpo_pairs.py output (train.jsonl / val.jsonl), "drive" or "hf_dataset" -------
DATA_SOURCE = "hf_dataset"
HF_DATASET_REPO = "carev01/ft-opus-distill"   # private
HF_DATASET_REVISION = "dpo1"
DRIVE_DATA_DIR = "/content/drive/MyDrive/ft-opus-dpo1"   # only when DATA_SOURCE == "drive"

# --- Google Drive: checkpoints, final adapter, metrics (survive a runtime death) ----------
RUN_NAME = "opus-distill-dpo2"     # dpo1 = the 5e-6 run (too weak, not exported)
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/graph-rag-distill") / RUN_NAME
CKPT_DIR = DRIVE_RUN_DIR / "checkpoints"
FINAL_ADAPTER_DIR = DRIVE_RUN_DIR / "final-adapter"
FINAL_MARKER = FINAL_ADAPTER_DIR / "COMPLETE"   # written LAST: adapter fully saved

# --- the model DPO starts from: the merged SFT checkpoint the pairs were generated with ---
HF_MODEL_REPO = "carev01/qwen35-4b-graphrag"
START_REVISION = "opus-distill-2026-10-ckpt100"
START_DIR = Path("/content/start-model")    # adapter_config points here: keep it fixed
MAX_SEQ_LENGTH = 20480
LOAD_IN_4BIT = True

# --- LoRA (same shape as the SFT run) ----------------------------------------------------
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# --- DPO -----------------------------------------------------------------------------------
BETA = 0.1                 # strength of the pull towards the reference (ckpt100)
NLL_ALPHA = 1.0            # NLL on chosen (rpo_alpha / "sft" loss): protects coverage; 0 = plain DPO
LEARNING_RATE = 5e-5       # dpo1's 5e-6 moved the split-point log-odds by only +0.34
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.1
LR_SCHEDULER_TYPE = "cosine"
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_STEPS = 10            # also the save interval, so the best checkpoint is on Drive
SAVE_TOTAL_LIMIT = 4
SEED = 20261021

# --- export ----------------------------------------------------------------------------------
HF_MODEL_REVISION = "opus-distill-2026-10-dpo2"
MIN_GREEDY_GAIN = 4        # export gate: val pairs where greedy now skips the removed edge
HF_MODEL_PRIVATE = False   # the repo is public by design; a branch cannot be more private

OUT_DIR = Path("/content/outputs/opus-distill-dpo")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("config set")

## 3a. Mount Google Drive

Same as `distill_train.ipynb` section 3a.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
_ckpts = sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print(f"run dir: {DRIVE_RUN_DIR}")
print(f"checkpoints on Drive: {[p.name for p in _ckpts] or 'none'}")
print(f"final adapter on Drive (complete): {FINAL_MARKER.exists()}")

## 4. Get the dataset

In [ ]:
DATA_DIR = Path("/content/data/ft-opus-dpo1")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "drive":
    DATA_DIR = Path(DRIVE_DATA_DIR)
elif DATA_SOURCE == "hf_dataset":
    from huggingface_hub import hf_hub_download
    for fname in ("train.jsonl", "val.jsonl", "MANIFEST.json", "sample.json"):
        try:
            p = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                                filename=fname, revision=HF_DATASET_REVISION, token=HF_TOKEN)
        except Exception:
            if fname in ("sample.json", "MANIFEST.json"):
                print(f"  {fname} not in {HF_DATASET_REPO}@{HF_DATASET_REVISION} -- skipping")
                continue
            raise
        (DATA_DIR / fname).write_bytes(Path(p).read_bytes())
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

assert (DATA_DIR / "train.jsonl").exists() and (DATA_DIR / "val.jsonl").exists(), DATA_DIR
print(f"dataset ready at {DATA_DIR}")

## 5. Load the pairs and check their shape

Every record: `messages` = `[system, user]` exactly as vLLM received them, `chosen` and
`rejected` = the reply JSON. `chosen` must parse, hold fewer edges than `rejected`, and
every chosen edge must appear in `rejected` unchanged.

In [ ]:
import json
from collections import Counter


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_records = read_jsonl(DATA_DIR / "train.jsonl")
val_records = read_jsonl(DATA_DIR / "val.jsonl")
for _r in train_records + val_records:
    assert [m["role"] for m in _r["messages"]] == ["system", "user"], _r["meta"]
    _c, _x = json.loads(_r["chosen"])["edges"], json.loads(_r["rejected"])["edges"]
    assert 0 < len(_c) < len(_x), _r["meta"]
    assert all(e in _x for e in _c), _r["meta"]
_cats = Counter(c for r in train_records for f in r["meta"]["removed"] for c in f["categories"][:1])
print(f"train pairs {len(train_records)}, val pairs {len(val_records)}")
print(f"facts removed (train): {sum(len(r['meta']['removed']) for r in train_records)}; "
      f"categories {dict(_cats)}")
assert len(train_records) >= 50, "too few pairs for a DPO round -- capture more first"

## 6. Download the starting model and attach LoRA

The merged bf16 weights of `START_REVISION` go to `START_DIR`. A fixed path matters: the
saved adapter records it as its base, and the export (section 10) reloads from there.

In [ ]:
from huggingface_hub import snapshot_download
from unsloth import FastLanguageModel
import torch

snapshot_download(repo_id=HF_MODEL_REPO, revision=START_REVISION, local_dir=str(START_DIR),
                  ignore_patterns=["*.gguf"], token=HF_TOKEN)
assert any(START_DIR.glob("*.safetensors")), f"no safetensors in {HF_MODEL_REPO}@{START_REVISION}"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(START_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print(f"{START_REVISION} loaded, LoRA attached (the reference model is the same weights "
      "with the adapter disabled)")
import transformers.models.qwen3_5.modeling_qwen3_5 as _mq


def calls_package(fn, package: str) -> bool:
    # Follow transformers' kernel wrappers down to the implementation they call.
    seen, stack = set(), [fn]
    while stack:
        f = stack.pop()
        if f is None or id(f) in seen or not callable(f):
            continue
        seen.add(id(f))
        if str(getattr(f, "__module__", "")).split(".")[0] == package:
            return True
        stack.append(getattr(f, "__wrapped__", None))
        for c in getattr(f, "__closure__", None) or ():
            try:
                stack.append(c.cell_contents)
            except ValueError:
                pass
    return False


_fast = {"causal_conv1d": calls_package(_mq.causal_conv1d_fn, "causal_conv1d"),
         "fla": calls_package(_mq.chunk_gated_delta_rule, "fla")}
print(f"fast kernels in use: {_fast}")
if CC1D_LOADED and not _fast["causal_conv1d"]:
    print("WARNING: the causal_conv1d kernel loaded in section 1a but the model does not "
          "call it -- this run is on the PyTorch path")

## 7. Build prompt / chosen / rejected text

`prompt` is the serving prompt: the chat template with `add_generation_prompt=True,
enable_thinking=False`, which already ends in the empty think block. Completions get
`<|im_end|>` unless this TRL version appends the end-of-turn token itself (section 8
decides). Pairs longer than `MAX_SEQ_LENGTH` are dropped, never truncated: a truncated
prompt would train on a prompt vLLM never sees.

In [ ]:
from datasets import Dataset

END_MARK = "<|im_end|>"
# Unsloth may hand back a multimodal processor for this architecture; its __call__ batches
# (len(ids) would be 1) and it has no eos_token. Count and decode with the text tokenizer.
TEXT_TOK = getattr(tokenizer, "tokenizer", tokenizer)
RESPONSE_PART = tokenizer.apply_chat_template(
    [{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True,
    enable_thinking=False).rsplit(END_MARK + "\n", 1)[-1]
print(f"serving prompt ends with {RESPONSE_PART!r}")


def to_pair(r: dict, mark: str) -> dict:
    prompt = tokenizer.apply_chat_template(r["messages"], tokenize=False,
                                           add_generation_prompt=True, enable_thinking=False)
    assert prompt.endswith(RESPONSE_PART), prompt[-80:]
    return {"prompt": prompt, "chosen": r["chosen"] + mark, "rejected": r["rejected"] + mark}


def build(records: list[dict], mark: str) -> Dataset:
    rows, dropped = [], 0
    for r in records:
        p = to_pair(r, mark)
        n = len(TEXT_TOK(p["prompt"] + p["rejected"], add_special_tokens=False)["input_ids"])
        if n > MAX_SEQ_LENGTH:
            dropped += 1
            continue
        rows.append(p)
    print(f"  {len(rows)} pairs kept, {dropped} over {MAX_SEQ_LENGTH} tokens dropped")
    return Dataset.from_list(rows)


_lens = sorted(len(TEXT_TOK(to_pair(r, "")["prompt"], add_special_tokens=False)["input_ids"])
               for r in train_records)
MAX_PROMPT_TOKENS = min(_lens[-1], MAX_SEQ_LENGTH)
print(f"prompt tokens: median {_lens[len(_lens) // 2]}, max {_lens[-1]}")

## 8. Build the trainer -- and check what it will actually train on (fail loudly)

TRL versions differ: some append the tokenizer's EOS to every completion, and some
tokenise the prompt with special tokens added. So this cell decodes the trainer's own
first tokenised row and requires prompt == serving prompt and chosen == answer +
`<|im_end|>` (a trailing newline is accepted). If this TRL appended `<|im_end|>` a
second time, the trainer is rebuilt once with the completions unmarked.

In [ ]:
import dataclasses
import inspect

try:  # older Unsloth needs the explicit patch; newer ones patch on import
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
except ImportError:
    pass
from trl import DPOConfig, DPOTrainer

_fields = {f.name for f in dataclasses.fields(DPOConfig)}
_extra = {}
if "max_prompt_length" in _fields:
    _extra["max_prompt_length"] = MAX_PROMPT_TOKENS
# transformers 5.x removed warmup_ratio; its warmup_steps reads a float < 1 as a ratio.
_extra["warmup_ratio" if "warmup_ratio" in _fields else "warmup_steps"] = WARMUP_RATIO
if NLL_ALPHA and "rpo_alpha" in _fields:      # TRL < 1.0
    _extra["rpo_alpha"] = NLL_ALPHA
elif NLL_ALPHA:
    # TRL 1.x dropped rpo_alpha. The same term is the "sft" loss: the mean per-token NLL
    # of the chosen completion, added to the sigmoid (DPO) loss with weight NLL_ALPHA.
    assert "loss_weights" in _fields, ("this TRL has neither rpo_alpha nor loss_weights: "
                                       "set NLL_ALPHA = 0 only if you accept plain DPO "
                                       "(coverage risk)")
    _extra["loss_type"] = ["sigmoid", "sft"]
    _extra["loss_weights"] = [1.0, NLL_ALPHA]
print(f"DPOConfig extras: {_extra}")

dpo_config = DPOConfig(
    output_dir=str(CKPT_DIR),
    beta=BETA,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim="adamw_8bit",
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_length=MAX_SEQ_LENGTH,
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    **_extra,
)
_tok_kw = ("processing_class" if "processing_class" in inspect.signature(DPOTrainer.__init__).parameters
           else "tokenizer")


def make_trainer(mark: str) -> DPOTrainer:
    # TRL 1.x copies the untrained adapter into a frozen "ref" adapter (= ckpt100, since
    # LoRA starts at zero) for the reference pass; a rebuild must not find one already.
    if "ref" in getattr(model, "peft_config", {}):
        model.delete_adapter("ref")
    print(f"building datasets (completion mark {mark!r})")
    return DPOTrainer(model=model, ref_model=None, args=dpo_config,
                      train_dataset=build(train_records, mark),
                      eval_dataset=build(val_records, mark), **{_tok_kw: tokenizer})


def first_row(t: DPOTrainer) -> tuple[str, str]:
    row = t.train_dataset[0]
    for p_col, c_col in (("prompt_input_ids", "chosen_input_ids"), ("prompt_ids", "chosen_ids")):
        if p_col in row and c_col in row:
            dec = lambda ids: TEXT_TOK.decode(ids, skip_special_tokens=False)
            return dec(row[p_col]), dec(row[c_col])
    raise AssertionError(f"unknown tokenised layout {sorted(row)} -- check this TRL version's "
                         "DPOTrainer before training")


trainer = make_trainer(END_MARK)
_p, _c = first_row(trainer)
_answer = train_records[0]["chosen"]
if _c.startswith(_answer + END_MARK + END_MARK) or _c.startswith(_answer + END_MARK + "\n" + END_MARK):
    print("TRL appends <|im_end|> itself -- rebuilding without our mark")
    trainer = make_trainer("")
    _p, _c = first_row(trainer)
_serving = to_pair(train_records[0], "")["prompt"]
assert _p == _serving, f"tokenised prompt != serving prompt: {_p[-120:]!r} vs {_serving[-120:]!r}"
_OK_TAILS = {END_MARK, END_MARK + "\n"}
_eos = TEXT_TOK.eos_token or ""
_tail = _c[len(_answer):] if _c.startswith(_answer) else None
assert _tail is not None, f"chosen does not start with the answer: {_c[:120]!r}"
if _tail not in _OK_TAILS:
    # An EOS other than <|im_end|> appended after it: vLLM stops at <|im_end|>, harmless.
    assert _eos and _eos != END_MARK and _tail.endswith(_eos) and _tail[:-len(_eos)] in _OK_TAILS, (
        f"chosen does not decode to answer + <|im_end|>: tail {_tail!r}")
    print(f"note: TRL also trains on {_eos!r} after <|im_end|> (harmless)")
print(f"OK: trains on serving prompt + answer + {_tail!r}")

## 9. Train (resumable), then evaluate on the held-out pairs

`eval_rewards/accuracies` is the share of held-out pairs where the model now prefers the
reply without the invented fact more than ckpt100 did. Look for it well above 0.5, and
for `eval_logps/chosen` not collapsing (that is the coverage risk).

In [ ]:
import json as _json

_CKPT_FILES = ("trainer_state.json", "optimizer.pt", "scheduler.pt")


def last_complete_checkpoint(ckpt_dir: Path) -> str | None:
    for c in sorted(ckpt_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]),
                    reverse=True):
        if all((c / f).exists() for f in _CKPT_FILES) and any(c.glob("adapter_model.*")):
            return str(c)
        print(f"skipping incomplete {c.name}")
    return None


if FINAL_MARKER.exists():
    print(f"final adapter already on Drive ({FINAL_ADAPTER_DIR}) -- skipping to export. "
          "Delete that folder to train again.")
else:
    _last = last_complete_checkpoint(CKPT_DIR)
    print(f"resuming from {_last}" if _last else "no checkpoint on Drive -- starting fresh")
    train_result = trainer.train(resume_from_checkpoint=_last)
    eval_metrics = trainer.evaluate()   # the best checkpoint (load_best_model_at_end)
    print(_json.dumps(eval_metrics, indent=2))
    model.save_pretrained(str(FINAL_ADAPTER_DIR), selected_adapters=["default"])  # not TRL's "ref"
    tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
    (DRIVE_RUN_DIR / "train_result.json").write_text(_json.dumps(train_result.metrics, indent=2))
    (DRIVE_RUN_DIR / "eval_metrics.json").write_text(_json.dumps(eval_metrics, indent=2))
    (DRIVE_RUN_DIR / "log_history.json").write_text(
        _json.dumps(trainer.state.log_history, indent=2))
    FINAL_MARKER.write_text("adapter + tokenizer fully saved\n")
    print(f"best checkpoint: {trainer.state.best_model_checkpoint}")
    _acc = eval_metrics.get("eval_rewards/accuracies")
    if _acc is not None and _acc < 0.6:
        print(f"WARNING: held-out reward accuracy {_acc:.2f} -- DPO barely separated the pairs")

## 10. Export -- merged bf16 safetensors to a NEW revision

Refuses a revision that already has its own commits. Reloads the DPO model (ckpt100 +
the final adapter) and ckpt100 itself, then two gates before anything is uploaded:

- **Behaviour changed.** For each val pair, at the first token where `chosen` and
  `rejected` differ (where the unsupported edge starts), compare the two continuations'
  log-probabilities. Greedy decoding must skip the edge in at least `MIN_GREEDY_GAIN`
  more pairs than ckpt100 does. Reward accuracy alone does not show this (dpo1: 0.97
  accuracy, zero pairs changed).
- **Still answers in JSON.** Three short held-out prompts, with both models' edge counts
  printed side by side: a large drop means DPO is cutting supported edges too.

Then it merges, pushes and checks the Hub's shards equal the merged index.

In [ ]:
import gc

from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
# A new branch is a copy of main, weights included, so "the branch holds weights" cannot
# be the test: refuse only a branch that has moved away from main (an earlier export).
_refs = {b.name: b.target_commit
         for b in api.list_repo_refs(HF_MODEL_REPO, repo_type="model").branches}
assert _refs.get(HF_MODEL_REVISION, _refs["main"]) == _refs["main"], (
    f"{HF_MODEL_REPO}@{HF_MODEL_REVISION} already has its own commits (an earlier export?)"
    " -- choose a new HF_MODEL_REVISION")
api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=HF_MODEL_REVISION,
                  exist_ok=True)

for _name in ("trainer", "model"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

assert FINAL_MARKER.exists(), f"no complete final adapter at {FINAL_ADAPTER_DIR}"
assert any(START_DIR.glob("*.safetensors")), "run section 6 first (the adapter's base)"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(FINAL_ADAPTER_DIR), max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, dtype=None)
FastLanguageModel.for_inference(model)
base, base_tok = FastLanguageModel.from_pretrained(
    model_name=str(START_DIR), max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT, dtype=None)
FastLanguageModel.for_inference(base)


def choice_at_split(m, r: dict) -> tuple[float, bool] | None:
    # (log p(chosen token) - log p(rejected token), greedy picks the chosen token) at the
    # first token where chosen and rejected differ: where the removed edge starts.
    p = to_pair(r, "")
    ch = TEXT_TOK(p["prompt"] + p["chosen"], add_special_tokens=False)["input_ids"]
    rj = TEXT_TOK(p["prompt"] + p["rejected"], add_special_tokens=False)["input_ids"]
    split = next((i for i, (a, b) in enumerate(zip(ch, rj)) if a != b), None)
    if split is None:
        return None
    ids = torch.tensor([ch[:split]], device=m.device)
    out = m.generate(input_ids=ids, attention_mask=torch.ones_like(ids), max_new_tokens=1,
                     do_sample=False, output_logits=True, return_dict_in_generate=True)
    lp = torch.log_softmax(out.logits[0][0].float(), -1)
    return lp[ch[split]].item() - lp[rj[split]].item(), int(lp.argmax()) == ch[split]


_skips = {}
for _label, _m in (("ckpt100", base), ("dpo", model)):
    _res = [x for x in (choice_at_split(_m, r) for r in val_records) if x]
    _odds = sorted(o for o, _ in _res)
    _skips[_label] = sum(g for _, g in _res)
    print(f"[{_label}] {len(_res)} pairs: greedy skips the removed edge in {_skips[_label]}, "
          f"mean log-odds {sum(_odds) / len(_odds):+.2f}, median {_odds[len(_odds) // 2]:+.2f}")
assert _skips["dpo"] - _skips["ckpt100"] >= MIN_GREEDY_GAIN, (
    f"DPO changed greedy behaviour in only {_skips['dpo'] - _skips['ckpt100']} val pairs "
    f"(gate {MIN_GREEDY_GAIN}) -- too weak to be worth exporting or evaluating")


def edges_of(m, t, prompt: str) -> int | None:
    ids = t(text=prompt, return_tensors="pt").to(m.device)
    out = m.generate(**ids, max_new_tokens=4096, do_sample=False)   # 1024 cut an 11-edge reply
    reply = t.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        return len(_json.loads(reply)["edges"])
    except (ValueError, KeyError, TypeError):
        print(f"  not JSON: {reply[:160]!r} ... {reply[-160:]!r}")
        return None


for _r in sorted(val_records, key=lambda r: len(r["messages"][1]["content"]))[:3]:
    _prompt = to_pair(_r, "")["prompt"]
    _n_dpo, _n_base = edges_of(model, tokenizer, _prompt), edges_of(base, base_tok, _prompt)
    print(f"edges: dpo {_n_dpo}, ckpt100 {_n_base}, chosen {len(_json.loads(_r['chosen'])['edges'])}")
    assert _n_dpo is not None, "the DPO model did not answer in JSON -- do not upload this export"
print("smoke test passed")
del base, base_tok
gc.collect()
torch.cuda.empty_cache()

merged_dir = OUT_DIR / "merged-safetensors"
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
_index = json.loads((merged_dir / "model.safetensors.index.json").read_text())
_missing = sorted(set(_index["weight_map"].values())
                  - {p.name for p in merged_dir.glob("*.safetensors")})
assert not _missing, f"merged save is missing shards: {_missing}"
api.upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(merged_dir),
                  revision=HF_MODEL_REVISION,
                  delete_patterns=["*.safetensors", "*.gguf"],   # no shard of main's survives
                  commit_message=f"{RUN_NAME}: DPO on {START_REVISION}, merged bf16")
_pushed = {f for f in api.list_repo_files(HF_MODEL_REPO, repo_type="model",
                                          revision=HF_MODEL_REVISION)
           if f.endswith((".safetensors", ".gguf"))}
assert _pushed == set(_index["weight_map"].values()), f"Hub shards != merged index: {_pushed}"
print(f"DONE: {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

## 11. Next -- outside this notebook

Hand the revision name back. It is validated like ckpt100, on the SAME 60 trial articles
(none of them in the DPO data): held-out eval (`scripts/eval_finetune.py --guided`), the
45-chunk faithfulness check, then `scripts/haiku_extract_ab.py --arms v2 --reuse
<adoption-trial rows>` against the adoption trial's routed rows, judged twice.